In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, f1_score

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, AdaBoostRegressor, AdaBoostClassifier
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier

from xgboost import XGBRegressor, XGBClassifier
from lightgbm import LGBMRegressor, LGBMClassifier

# ---------------- LOAD DATA ----------------
df = pd.read_csv("/home/midfielder/PythonEnvironment/projects/ml/first_project/student_academic_performance.csv")



In [9]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, f1_score
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier


# 2. FIX: FORCE NUMERIC SCORES (Stop them from becoming 0/1)
score_cols_raw = ['Math Score', 'Reading Score', 'Writing Score']
for col in score_cols_raw:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 3. BASIC CLEANING
df = df.drop_duplicates()
if "Lunch Type" in df.columns:
    df = df.drop(columns=["Lunch Type"])

# 4. CUSTOM ENCODING (Gender & Education)
def encode_custom(df_in):
    df_c = df_in.copy()
    if "Gender" in df_c.columns:
        df_c["Gender"] = df_c["Gender"].map({"male": 0, "female": 1})
    if "Parental Education" in df_c.columns:
        order = {
            "some high school": 0, "high school": 1, "some college": 2,
            "associate's degree": 3, "bachelor's degree": 4, "master's degree": 5
        }
        df_c["Parental Education"] = df_c["Parental Education"].map(order)
    
    # One-Hot Encoding for Race/Ethnicity, School Type, and Internet
    one_hot_cols = [c for c in ["Race/Ethnicity", "School Type", "Internet Access"] if c in df_c.columns]
    df_c = pd.get_dummies(df_c, columns=one_hot_cols, drop_first=True)
    return df_c

df = encode_custom(df)

# 5. SANITIZATION
def sanitize_columns(df_in):
    df_in.columns = [re.sub(r'[^\w\s]', '', col).replace(' ', '_') for col in df_in.columns]
    return df_in

df = sanitize_columns(df)

# 6. SAFE NUMERIC CHECK (Ensures Math_Score etc. are NOT factorized)
sanitized_score_cols = ['Math_Score', 'Reading_Score', 'Writing_Score']
for col in df.columns:
    # Only factorize if it's an object/string AND not a score column
    if col not in sanitized_score_cols and df[col].dtype == 'object':
        df[col] = pd.factorize(df[col])[0]

# 7. TRAIN-TEST SPLIT
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# 8. MISSING INDICATORS (Renamed for clarity)
for col in list(train_df.columns):
    if train_df[col].isnull().sum() > 0:
        indicator_name = f"{col}_IS_MISSING"
        train_df[indicator_name] = train_df[col].isnull().astype(int)
        test_df[indicator_name] = test_df[col].isnull().astype(int)

# 9. MULTIMODAL IMPUTATION (MICE)
missing_cols = [col for col in train_df.columns if train_df[col].isnull().any()]

for col in missing_cols:
    # Regression for Scores, Classification for Categorical gaps
    is_clf = (train_df[col].nunique() < 10) and (col not in sanitized_score_cols)
    model = RandomForestClassifier() if is_clf else RandomForestRegressor()
    
    # Train on non-missing rows
    valid_data = train_df[train_df[col].notnull()]
    X_imp = valid_data.drop(columns=[col])
    y_imp = valid_data[col]
    
    # Temporary imputer for feature inputs
    temp_imp = SimpleImputer(strategy='mean')
    X_imp_filled = temp_imp.fit_transform(X_imp)
    
    model.fit(X_imp_filled, y_imp)
    
    # Fill gaps in both sets
    for dset in [train_df, test_df]:
        mask = dset[col].isnull()
        if mask.any():
            X_miss = temp_imp.transform(dset[mask].drop(columns=[col]))
            dset.loc[mask, col] = model.predict(X_miss)

# 10. CREATE TARGET & DROP LEAKAGE
train_df['Avg'] = train_df[sanitized_score_cols].mean(axis=1)
test_df['Avg'] = test_df[sanitized_score_cols].mean(axis=1)

def classify(score):
    if score >= 80: return 2  # High
    elif score >= 50: return 1 # Average
    else: return 0           # Low

train_df['Target'] = train_df['Avg'].apply(classify)
test_df['Target'] = test_df['Avg'].apply(classify)

# FINAL DROP: Remove scores and intermediate average
leakage_cols = sanitized_score_cols + ['Avg', 'Student_ID']
X_train = train_df.drop(columns=leakage_cols + ['Target'])
X_test = test_df.drop(columns=leakage_cols + ['Target'])
y_train = train_df['Target']
y_test = test_df['Target']

# 11. SAVE THE FILES
X_train.to_csv("X_train_final.csv", index=False)
X_test.to_csv("X_test_final.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("✅ Success! Check X_train_final.csv — scores are removed and data is clean.")

✅ Success! Check X_train_final.csv — scores are removed and data is clean.


In [10]:
# --- 14. SAVE FINAL PROCESSED DATA ---

# Option A: Save X and y separately (Best for Model Training)
X_train.to_csv("X_train_final.csv", index=False)
X_test.to_csv("X_test_final.csv", index=False)
y_train.to_csv("y_train_labels.csv", index=False)
y_test.to_csv("y_test_labels.csv", index=False)

# Option B: Save a single combined "Cleaned" file for your Research Paper
# We recombine X and y so you have a single table with the Target
train_full = X_train.copy()
train_full['Target'] = y_train

test_full = X_test.copy()
test_full['Target'] = y_test

final_cleaned_data = pd.concat([train_full, test_full], axis=0)

# Save the final version
final_cleaned_data.to_csv("student_performance_cleaned_final.csv", index=False)

print("\n💾 Files Saved Successfully:")
print("- X_train_final.csv / X_test_final.csv (Features)")
print("- y_train_labels.csv / y_test_labels.csv (Targets)")
print("- student_performance_cleaned_final.csv (Full Dataset)")


💾 Files Saved Successfully:
- X_train_final.csv / X_test_final.csv (Features)
- y_train_labels.csv / y_test_labels.csv (Targets)
- student_performance_cleaned_final.csv (Full Dataset)


In [11]:
# Check the range of values in your new final file
print(f"Unique Target Values: {final_cleaned_data['Target'].unique()}") # Should be [0, 1, 2]
print(f"Number of Columns: {len(final_cleaned_data.columns)}")

Unique Target Values: [1 2 0]
Number of Columns: 18
